# Groundwork notebook

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ.pop("HF_TOKEN", None)          
os.environ.pop("HUGGINGFACE_TOKEN", None)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
GEN_MODEL = "openai/gpt-oss-20b"
JUDGE_MODEL = "openai/gpt-oss-120b"
EMBED_MODEL = "all-MiniLM-L6-v2"

## 1. RAG : load, chunk, embed, store

In [2]:
from langchain_core.embeddings import Embeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer

docs = DirectoryLoader(
    "knowledge_base", glob="*.txt", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).load()
print(f"loaded {len(docs)} documents")

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(docs)
print(f"created {len(chunks)} chunks")


class LocalEmbeddings(Embeddings):
    """Local sentence-transformers embeddings (no HuggingFace login needed)."""

    def __init__(self, name):
        self.model = SentenceTransformer(name, use_auth_token=False)
        self.model_name = name

    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text):
        return self.model.encode(text, normalize_embeddings=True).tolist()


embeddings = LocalEmbeddings(EMBED_MODEL)
store = Chroma.from_documents(chunks, embedding=embeddings)

C:\Users\Admin\AppData\Local\Temp\ipykernel_9548\1266713573.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
c:\Users\Admin\enterprise-knowledge-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded 10 documents
created 49 chunks


c:\Users\Admin\enterprise-knowledge-assistant\.venv\Lib\site-packages\sentence_transformers\sentence_transformer\model.py:178: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in a future release of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1592.76it/s]


## 2. Retrieve context for a question

In [3]:
question = "How many days of paid annual leave do full-time employees get each year?"

hits = store.similarity_search(question, k=4)
contexts = [d.page_content for d in hits]
context_block = "\n\n".join(contexts)
print(context_block[:500])

Employees must submit planned annual leave requests at least **14 calendar days in advance** through the People Portal. Requests are approved by the employee's direct manager, and managers should respond within 3 business days.

### Carryover and Payout

A maximum of **5 unused annual leave days** may be carried over into the following calendar year. Carried-over days must be used by **March 31**, after which they expire. Unused annual leave is not paid out except upon termination of employment,


## 3. Generate a grounded answer

In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model=GEN_MODEL, api_key=GROQ_API_KEY, temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. If the answer is not there, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

answer = (prompt | llm).invoke({"context": context_block, "question": question}).content
print(answer)

Full‑time employees receive **22 days of paid annual leave** each calendar year.


## 4. Score the answer with RAGAS

In [11]:
import nest_asyncio
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    ContextEntityRecall,
    AnswerSimilarity,
    AnswerCorrectness,
)

nest_asyncio.apply()
sample = {
    "user_input": "How long is customer account data retained after an account is closed?",
    "response": "Customer account data is retained for 7 years after the account is closed.",
    "retrieved_contexts": [
        "Customer account data: retained for 7 years after account closure."
    ],
    "reference": "Customer account data is retained for 7 years after account closure."
}
sample = SingleTurnSample(
    user_input=sample["user_input"],
    response=sample["response"],
    retrieved_contexts=sample["retrieved_contexts"],
    reference=sample["reference"]
)

judge = LangchainLLMWrapper(ChatGroq(model=JUDGE_MODEL, api_key=GROQ_API_KEY, temperature=0))
scores = evaluate(
    dataset=EvaluationDataset(samples=[sample]),
    metrics = [
    Faithfulness(),
    AnswerRelevancy(),
    ContextPrecision(),
    ContextRecall(),
    ContextEntityRecall(),
    AnswerSimilarity(),
    AnswerCorrectness(),
    ],
    llm=judge,
    embeddings=LangchainEmbeddingsWrapper(embeddings),
)
print(scores)

C:\Users\Admin\AppData\Local\Temp\ipykernel_9548\350074349.py:31: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge = LangchainLLMWrapper(ChatGroq(model=JUDGE_MODEL, api_key=GROQ_API_KEY, temperature=0))
C:\Users\Admin\AppData\Local\Temp\ipykernel_9548\350074349.py:44: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings=LangchainEmbeddingsWrapper(embeddings),
Evaluating:  14%|█▍        | 1/7 [00:01<00:08,  1.49s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised

{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_precision': 1.0000, 'context_recall': 1.0000, 'context_entity_recall': 1.0000, 'answer_similarity': nan, 'answer_correctness': nan}


In [16]:
print(scores)

{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_precision': 1.0000, 'context_recall': 1.0000, 'context_entity_recall': 1.0000, 'answer_similarity': nan, 'answer_correctness': nan}


In [13]:
type(scores)

ragas.dataset_schema.EvaluationResult